# STC Jawwy

In [ ]:
"""
Here we install libraries that are not installed by default
Example:  pyslsb
Feel free to add any library you are planning to use.
"""
!pip install pyxlsb

In [ ]:
# Import the required libraries
"""
Please feel free to import any required libraries as per your needs
"""
import pandas as pd     # provides high-performance, easy to use structures and data analysis tools
import pyxlsb           # Excel extention to read xlsb files (the input file)
import numpy as np      # provides fast mathematical computation on arrays and matrices

# Jawwy dataset
The dataset consists of meta details about the movies and tv shows as genre.
Also details about Users activities, spent duration and if watching in High definition or standard definition.
You have to analyse this dataset to find top insights, findings and to solve the four tasks assigned to you.

In [ ]:
dataframe = pd.read_excel("stc TV Data Set_T1.xlsb",sheet_name="Final_Dataset")
# Please make a copy of dataset if you are going to work directly and make changes on the dataset
df=dataframe.copy()

# Task 1
##### You are required to work on task one to study and HD flag for available dataset

In [ ]:
# make a copy of the dataframe for working on task 1
df=dataframe.copy()

In [ ]:
grouped = df.copy()

# Make each episode unique by appending season/episode info
grouped.loc[grouped['program_class'] == 'SERIES/EPISODES', 'program_name'] = (
    grouped['program_name']
    + '_SE' + grouped['season'].astype(str)
    + '_EP' + grouped['episode'].astype(str)
)

# Aggregate metrics by program
grouped = (
    grouped.groupby(['program_name', 'program_class'])
    .agg({
        'user_id_maped': [
            ('No of Users who Watched', 'nunique'),
            ('No of watches', 'count')
        ],
        'duration_seconds': [
            ('Total watch time in seconds', 'sum')
        ]
    })
    .reset_index()
)

# Flatten column names
grouped.columns = [
    'program_name',
    'program_class',
    'No of Users who Watched',
    'No of watches',
    'Total watch time in seconds'
]

# Convert watch time to hours
grouped['Total watch time in hours'] = (
    grouped['Total watch time in seconds'] / 3600
)

# Remove seconds column
grouped = grouped.drop(columns=['Total watch time in seconds'])

# Rank by watch time, then watches, then unique viewers
grouped = (
    grouped.sort_values(
        by=[
            'Total watch time in hours',
            'No of watches',
            'No of Users who Watched'
        ],
        ascending=False
    )
    .reset_index(drop=True)
)


In [ ]:
# show the result
grouped.head(35)

,program_name,program_class,No of Users who Watched,No of watches,Total watch time in houres
0,The Boss Baby,MOVIE,3389,24047,2961.350833
1,The Amazing pider-Man,MOVIE,1011,2877,1966.119167
2,The Expendables,MOVIE,853,2119,1961.159444
3,Moana,MOVIE,2173,8081,1706.176944
4,Trolls,MOVIE,2613,13793,1601.023056
5,Bean,MOVIE,949,3617,1423.955000
6,The murfs,MOVIE,867,3132,1342.141111
7,Hotel Transylvania,MOVIE,491,1947,1096.533611
8,Cloudy With a Chance of Meatballs,MOVIE,683,2076,948.674722
9,The Man With The Iron Fists,MOVIE,707,2505,859.626389


In [ ]:
# we import Visualization libraries
# you can ignore and use any other graphing libraries
import matplotlib.pyplot as plt # a comprehensive library for creating static, animated, and interactive visualizations
import plotly #a graphing library makes interactive, publication-quality graphs. Examples of how to make line plots, scatter plots, area charts, bar charts, error bars, box plots, histograms, heatmaps, subplots, multiple-axes, polar charts, and bubble charts.
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
fig = px.bar(
    grouped.head(10),
    x='program_name',
    y='Total watch time in hours',
    color='program_class',
    title='Top 10 Programs by Total Watch Time'
)

fig.show()

In [ ]:
# Here we try to study the customer experience against Program class
grouped=df.copy()
grouped = grouped.groupby('program_class')\
.agg({'user_id_maped': [('co1', 'nunique'),('co2', 'count')],\
      'duration_seconds': [('co3', 'sum')] }).reset_index()
grouped.columns = ['program_class','No of Users who Watched', 'No of watches', 'Total watch time in seconds']
grouped['Total watch time in houres']=grouped['Total watch time in seconds']/3600
grouped = grouped.drop(columns=['Total watch time in seconds'])
grouped = grouped.sort_values(by=['Total watch time in houres', 'No of watches','No of Users who Watched'], ascending=False).reset_index(drop=True)


In [ ]:
# show the result
grouped.head()

,program_class,No of Users who Watched,No of watches,Total watch time in houres
0,SERIES/EPISODES,3901,560174,255097.787500
1,MOVIE,11355,488401,103444.145556


In [ ]:
# plot the total watch time against total number of users and report your findings
fig = px.pie(grouped, values='Total watch time in houres', names='program_class',\
             hover_data=['program_class'],title='Total duration spent by program_class')
fig2 = px.pie(grouped, values='No of Users who Watched', names='program_class',\
             hover_data=['program_class'],title='Total Users watching by program_class')

fig.update_traces(sort=False)
fig2.update_traces(sort=False)
fig.show()
fig2.show()

In [ ]:
import plotly.express as px

df_quality = df.copy()

# Convert 0/1 to labels
df_quality['Quality'] = df_quality['hd'].map({1: 'HD', 0: 'SD'})

# Group by new labels
quality_grouped = (
    df_quality.groupby('Quality')['user_id_maped']
    .nunique()
    .reset_index()
)

quality_grouped.columns = ['Quality', 'No of Users']

# Pie chart
fig = px.pie(
    quality_grouped,
    values='No of Users',
    names='Quality',
    hover_data=['Quality'],
    title='Users Watching: HD vs SD'
)

fig.update_traces(sort=False)
fig.show()